In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.research_config import ResearchConfig


# 08 Drawdown Diagnostics

Diagnose the Module 07 portfolio’s drawdown and realized trade outcomes. No model parameters or trading rules are refitted.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Settings


In [ ]:
cfg = ResearchConfig().validate()


## 2. Load portfolio files and exact drawdown

Include the initial-capital reference in the running peak.


In [ ]:
trades = pd.read_parquet("trades.parquet")
equity = pd.read_parquet("equity_curve.parquet")
running_peak = equity.equity.cummax().clip(lower=cfg.initial_capital)
drawdown = equity.equity / running_peak - 1
trough = drawdown.idxmin()
peak = equity.loc[:trough, "equity"].idxmax()
drawdown.to_frame("drawdown").to_parquet("drawdown_series.parquet")
pd.to_pickle(
    {"peak_date": peak, "trough_date": trough, "max_drawdown": drawdown.min()},
    "drawdown_episode.pkl",
)
display(pd.Series({"peak_date": peak, "trough_date": trough, "max_drawdown": drawdown.min()}))
drawdown.plot(figsize=(11, 3), title="Portfolio drawdown")
plt.show()


## 3. Worst days and loss clustering

Daily changes include option marks. Exit-date realized PnL is a different quantity.


In [ ]:
daily_changes = equity.equity.diff().to_frame("equity_change")
display(daily_changes.nsmallest(10, "equity_change"))
if not trades.empty:
    losses_by_exit = trades.groupby("exit_date").pnl.sum().to_frame("realized_pnl")
    losses_by_exit.to_parquet("pnl_by_exit_date.parquet")
    display(losses_by_exit.nsmallest(10, "realized_pnl"))
else:
    print("No completed trades.")


## 4. Pair performance and drawdown episode

Trade PnL for positions overlapping the episode is full-trade PnL, not an exact attribution of mark-to-market loss within the episode.


In [ ]:
if not trades.empty:
    pair_performance = trades.groupby("pair").agg(
        n_trades=("pnl", "size"),
        total_pnl=("pnl", "sum"),
        mean_trade_return=("trade_return", "mean"),
        win_rate=("pnl", lambda x: (x > 0).mean()),
    )
    overlapping = trades.loc[(trades.entry_date <= trough) & (trades.exit_date >= peak)].copy()
    pair_performance.to_parquet("pair_performance.parquet")
    overlapping.to_parquet("drawdown_overlapping_trades.parquet")
    display(pair_performance.sort_values("total_pnl").head(15))
    display(trades.nsmallest(10, "pnl")[["pair", "entry_date", "exit_date", "pnl", "trade_return"]])
equity[["n_open_positions"]].plot(figsize=(10, 3), title="Number of open pairs")
plt.show()
